# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [4]:
import pandas as pd
import datetime as dt
import numpy as np
pth = "orders.csv"
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
def clean(path):
    df = pd.read_csv(path) 
    print(f"STEP 1 {len(df)}")
    df["amount"] = pd.to_numeric(df["amount"].str.strip().str.replace(" ",""))
    print(f"STEP 2 {len(df)}")
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print(f"STEP 3 {len(df)}")
    df = df.drop_duplicates()
    print(f"STEP 4 {len(df)}")
    df["order_date"] = pd.to_datetime(df["order_date"])
    print(f"STEP 5 {len(df)}")
    df = df[df["customer_id"] !="TEST"]
    print(f"STEP 6 {len(df)}")
    print(f"Пустые Nan значения в amount: {df["amount"].isna().sum()}")
    print(f"Пустые Nan значения в city: {df["city"].isna().sum()}")
    print(f"Уникален ли order_ {df["order_id"].is_unique}")
    return df
    

In [6]:
pth = "orders.csv"
df = clean(pth)
len(df)

STEP 1 5463
STEP 2 5463
STEP 3 5463
STEP 4 5283
STEP 5 5283
STEP 6 5280
Пустые Nan значения в amount: 0
Пустые Nan значения в city: 0
Уникален ли order_ True


5280

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [14]:
sep = df[(df["status"] == "оплачен")&(df["order_date"].dt.month ==9)]["amount"].sum()
sep

np.float64(35243410.0)

In [13]:
oct = df[(df["status"] == "оплачен")&(df["order_date"].dt.month ==10)]["amount"].sum()
oct

np.float64(34419520.0)

In [18]:
print(f"Выручка оплаченных заказов за сентябрь: { sep} и октябрь: {sep} и изменение в %: {round(oct/sep*100 - 100,2)}")

Выручка оплаченных заказов за сентябрь: 35243410.0 и октябрь: 35243410.0 и изменение в %: -2.34


In [26]:
#Затем то же для каждого из 6 городов (цикл по городам)
for c in df["city"].unique():
    sep = df[(df["status"] == "оплачен")&(df["order_date"].dt.month ==9)&(df["city"] == c) ]["amount"].sum()
    oct = df[(df["status"] == "оплачен")&(df["order_date"].dt.month ==10)&(df["city"] == c)]["amount"].sum()
    print(f"Выручка оплаченных заказов в {c} за сентябрь: { sep} и октябрь: {sep} и изменение в %: {round(oct/sep*100 - 100,2)}%")

Выручка оплаченных заказов в Алматы за сентябрь: 14501070.0 и октябрь: 14501070.0 и изменение в %: 0.98%
Выручка оплаченных заказов в Астана за сентябрь: 9034220.0 и октябрь: 9034220.0 и изменение в %: -2.03%
Выручка оплаченных заказов в Караганда за сентябрь: 3648520.0 и октябрь: 3648520.0 и изменение в %: -7.59%
Выручка оплаченных заказов в Шымкент за сентябрь: 4558650.0 и октябрь: 4558650.0 и изменение в %: -11.29%
Выручка оплаченных заказов в Атырау за сентябрь: 923990.0 и октябрь: 923990.0 и изменение в %: 4.96%
Выручка оплаченных заказов в Актобе за сентябрь: 2576960.0 и октябрь: 2576960.0 и изменение в %: -1.42%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [ ]:
# 3

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [27]:
df["delivery_days"].isna().sum()

np.int64(158)

In [45]:
df[df["delivery_days"].isna()][["city","status"]].value_counts()

city       status 
Алматы     оплачен    61
Астана     оплачен    35
Караганда  оплачен    23
Шымкент    оплачен    17
Астана     отменён     5
Шымкент    отменён     5
Актобе     оплачен     4
Атырау     оплачен     4
Алматы     отменён     2
Атырау     отменён     1
Караганда  отменён     1
Name: count, dtype: int64

In [50]:
#Доля доставки за 2 дня в Шымкенте: (а) dropna, (б) fillna(медиана), (в) fillna(0). Какой вариант честный и почему другие вводят в заблуждение?
d = df.loc[df["city"] =="Шымкент","delivery_days"]
d <=2

5        True
19       True
25       True
30       True
31       True
        ...  
5409    False
5425    False
5428    False
5441    False
5448    False
Name: delivery_days, Length: 776, dtype: bool

In [51]:
(d<=2).sum() # кол-во заказов, у которых дсотавка длилась меньше 2 дней включительно

np.int64(225)

In [57]:
len(d)

776

In [64]:
225/776

0.28994845360824745

In [63]:
(d<=2).mean()

np.float64(0.28994845360824745)

In [61]:
#(а) dropna
d.isna().sum()

np.int64(22)

In [65]:
(d.dropna()<=2).mean()

np.float64(0.2984084880636605)

In [66]:
print(f"fillna(медиана): {(d.fillna(d.median()) <= 2).mean():.1%}")

fillna(медиана): 29.0%


In [69]:
print(f"fillna(0): {(d.fillna(0) <= 2).mean():.1%}")

fillna(0): 31.8%


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8